Celda 1. Instalar e importar librerias

In [1]:
# ============================================================
# GineAssist v1.0
# Chatbot basado en reglas para un consultorio de ginecología
# Python + NLTK
# ============================================================

!pip -q install nltk

import nltk
import random
import re
import unicodedata
from datetime import datetime

# Recursos necesarios de NLTK
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("GineAssist - Librerías cargadas correctamente")

GineAssist - Librerías cargadas correctamente


Celda 2. Informacion general del consultorio

In [4]:
consultorio = {
    "nombre": "Femicare",
    "telefono": "33-1655-1408",
    "direccion": "Centro Médico Puerta de Hierro Sur. Piso 7, consultorio 4A.",
    "horario": "lunes a jueves de 17:00 a 21:00",
    "costo_consulta": "$1,200 MXN",
    "metodos_pago": "efectivo y tarjeta",
    "servicios": [
        "consulta ginecológica",
        "revisión ginecológica preventiva",
        "orientación sobre servicios del consultorio"
    ]
}

Celda 3. Funciones de procesamiento con NLTK

In [5]:
# ============================================================
# PREPROCESAMIENTO DEL TEXTO
# ============================================================

def eliminar_acentos(texto):
    """
    Elimina acentos para facilitar la comparación
    entre las palabras introducidas por el usuario
    y las reglas del chatbot.
    """
    texto = unicodedata.normalize("NFD", texto)
    texto = "".join(
        caracter for caracter in texto
        if unicodedata.category(caracter) != "Mn"
    )
    return texto


def normalizar_texto(texto):
    """
    Convierte el texto a minúsculas, elimina acentos,
    signos innecesarios y espacios duplicados.
    """
    texto = texto.lower().strip()
    texto = eliminar_acentos(texto)

    # Conservamos letras y números
    texto = re.sub(r"[^a-z0-9ñ\s]", " ", texto)

    # Elimina espacios duplicados
    texto = re.sub(r"\s+", " ", texto)

    return texto.strip()


def tokenizar(texto):
    """
    Tokeniza el texto utilizando NLTK.
    """
    texto_normalizado = normalizar_texto(texto)
    tokens = nltk.word_tokenize(
        texto_normalizado,
        language="spanish"
    )
    return tokens

Celda 4. Definicion de intenciones

In [6]:
# ============================================================
# INTENCIONES DEL CHATBOT
# ============================================================

intenciones = {

    "saludo": {
        "palabras": [
            "hola", "buenos", "buenas",
            "hey", "saludos"
        ],
        "frases": [
            "buenos dias",
            "buenas tardes",
            "buenas noches"
        ],
        "respuestas": [
            "Hola. Soy GineAssist. ¿En qué puedo ayudarte?",
            "Hola. Bienvenido a GineAssist. Puedo ayudarte con información general del consultorio.",
            "Hola. Puedo orientarte sobre horarios, ubicación, servicios, citas, costos y requisitos."
        ]
    },

    "horario": {
        "palabras": [
            "horario", "horarios", "abren",
            "cierran", "atienden", "hora"
        ],
        "frases": [
            "a que hora",
            "horario de atencion",
            "cuando abren",
            "cuando cierran"
        ],
        "respuestas": [
            f"Nuestro horario de atención es {consultorio['horario']}."
        ]
    },

    "ubicacion": {
        "palabras": [
            "direccion", "ubicacion",
            "ubicados", "ubicada", "llegar"
        ],
        "frases": [
            "donde estan",
            "donde se encuentran",
            "como llego"
        ],
        "respuestas": [
            f"El consultorio se encuentra en {consultorio['direccion']}."
        ]
    },

    "servicios": {
        "palabras": [
            "servicio", "servicios",
            "ofrecen", "especialidades"
        ],
        "frases": [
            "que servicios",
            "que ofrecen",
            "que consultas"
        ],
        "respuestas": [
            "Entre nuestros servicios se encuentran: "
            + ", ".join(consultorio["servicios"])
            + ". Para conocer si un servicio específico está disponible, puedes comunicarte con recepción."
        ]
    },

    "cita": {
        "palabras": [
            "cita", "agendar", "reservar",
            "consulta", "appointment"
        ],
        "frases": [
            "hacer una cita",
            "agendar una cita",
            "quiero una cita",
            "solicitar una cita"
        ],
        "respuestas": [
            f"Para solicitar una cita puedes comunicarte con recepción al {consultorio['telefono']}."
        ]
    },

    "costo": {
        "palabras": [
            "costo", "cuesta", "precio",
            "cuanto", "tarifa"
        ],
        "frases": [
            "cuanto cuesta",
            "precio de consulta",
            "costo de consulta"
        ],
        "respuestas": [
            f"Para este proyecto académico, el costo de consulta registrado es {consultorio['costo_consulta']}. "
            "Te recomiendo confirmar el precio con recepción antes de tu cita."
        ]
    },

    "pago": {
        "palabras": [
            "pago", "pagos", "tarjeta",
            "efectivo", "pagar"
        ],
        "frases": [
            "formas de pago",
            "metodos de pago",
            "aceptan tarjeta"
        ],
        "respuestas": [
            f"Los métodos de pago registrados son {consultorio['metodos_pago']}."
        ]
    },

    "requisitos": {
        "palabras": [
            "llevar", "documentos",
            "requisitos", "necesito"
        ],
        "frases": [
            "que debo llevar",
            "que necesito llevar",
            "requisitos para consulta"
        ],
        "respuestas": [
            "Los requisitos pueden variar según el tipo de consulta. "
            f"Para confirmar qué debes llevar, comunícate con recepción al {consultorio['telefono']}."
        ]
    },

    "preparacion": {
        "palabras": [
            "preparar", "preparacion",
            "prepararme", "antes"
        ],
        "frases": [
            "como me preparo",
            "prepararme para la consulta",
            "antes de mi cita"
        ],
        "respuestas": [
            "La preparación puede variar dependiendo del servicio. "
            f"Para recibir las indicaciones correspondientes a tu cita, comunícate con recepción al {consultorio['telefono']}."
        ]
    },

    "reprogramacion": {
        "palabras": [
            "cambiar", "cancelar",
            "reprogramar", "modificar"
        ],
        "frases": [
            "cambiar mi cita",
            "cancelar mi cita",
            "reprogramar mi cita"
        ],
        "respuestas": [
            f"Para cambiar, cancelar o reprogramar una cita, comunícate con recepción al {consultorio['telefono']}."
        ]
    },

    "contacto": {
        "palabras": [
            "telefono", "contacto",
            "recepcion", "llamar"
        ],
        "frases": [
            "hablar con recepcion",
            "numero de telefono",
            "quiero hablar con alguien"
        ],
        "respuestas": [
            f"Puedes comunicarte con recepción al {consultorio['telefono']}."
        ]
    },

    "despedida": {
        "palabras": [
            "adios", "bye", "gracias",
            "salir", "terminar"
        ],
        "frases": [
            "muchas gracias",
            "hasta luego",
            "eso es todo"
        ],
        "respuestas": [
            "Con gusto. Gracias por utilizar GineAssist.",
            "Gracias por comunicarte con GineAssist. Que tengas un buen día.",
            "Ha sido un gusto orientarte. Hasta luego."
        ]
    }
}

Celda 5. Reglas de seguridad y delimitacion

In [7]:
# ============================================================
# REGLAS PARA CONSULTAS FUERA DEL ALCANCE
# ============================================================

terminos_clinicos_fuera_alcance = [
    "diagnostico",
    "diagnosticar",
    "medicamento",
    "medicina",
    "recetar",
    "receta",
    "tratamiento",
    "dosis",
    "resultado",
    "resultados",
    "laboratorio",
    "interpretar"
]

terminos_urgencia = [
    "emergencia",
    "urgencia",
    "urgente"
]


def detectar_consulta_fuera_alcance(tokens):
    """
    Identifica solicitudes médicas que requieren atención profesional.
    """
    for palabra in tokens:
        if palabra in terminos_clinicos_fuera_alcance:
            return True
    return False


def detectar_urgencia(tokens):
    """
    Detecta menciones explícitas de emergencia o urgencia.
    """
    for palabra in tokens:
        if palabra in terminos_urgencia:
            return True
    return False

Celda 6. Motor de deteccion de intenciones

In [8]:
# ============================================================
# MOTOR DE REGLAS
# ============================================================

def detectar_intencion(texto):
    """
    Identifica la intención del usuario mediante reglas
    basadas en palabras clave y frases.
    """

    texto_normalizado = normalizar_texto(texto)
    tokens = tokenizar(texto)

    mejor_intencion = None
    mejor_puntuacion = 0

    for nombre_intencion, datos in intenciones.items():

        puntuacion = 0

        # Coincidencias de palabras
        for palabra in datos["palabras"]:
            if palabra in tokens:
                puntuacion += 1

        # Las frases completas reciben mayor peso porque
        # proporcionan más contexto que una sola palabra.
        for frase in datos["frases"]:
            frase_normalizada = normalizar_texto(frase)

            if frase_normalizada in texto_normalizado:
                puntuacion += 3

        # Guardar la intención con mayor puntuación
        if puntuacion > mejor_puntuacion:
            mejor_puntuacion = puntuacion
            mejor_intencion = nombre_intencion

    return mejor_intencion, mejor_puntuacion

Celda 7. Generacion de respuesta

In [9]:
# ============================================================
# GENERADOR DE RESPUESTAS
# ============================================================

def generar_respuesta(texto):
    """
    Procesa el mensaje del usuario y genera
    una respuesta de GineAssist.
    """

    if not texto or not texto.strip():
        return (
            "No recibí ninguna pregunta. "
            "Puedes preguntarme sobre horarios, ubicación, "
            "servicios, citas, costos o requisitos."
        ), "entrada_vacia"

    tokens = tokenizar(texto)

    # Prioridad 1: menciones explícitas de urgencia
    if detectar_urgencia(tokens):
        respuesta = (
            "GineAssist no está diseñado para atender emergencias médicas. "
            "Si consideras que se trata de una emergencia, busca atención "
            "médica profesional inmediata a través de los servicios "
            "correspondientes de tu localidad."
        )
        return respuesta, "urgencia"

    # Prioridad 2: consultas clínicas fuera del alcance
    if detectar_consulta_fuera_alcance(tokens):
        respuesta = (
            "No puedo realizar diagnósticos, recomendar medicamentos, "
            "modificar tratamientos ni interpretar resultados médicos. "
            "Para este tipo de consulta debes comunicarte directamente "
            "con un profesional de la salud."
        )
        return respuesta, "fuera_de_alcance"

    # Prioridad 3: detección normal de intención
    intencion, puntuacion = detectar_intencion(texto)

    if intencion is not None and puntuacion > 0:
        respuesta = random.choice(
            intenciones[intencion]["respuestas"]
        )
        return respuesta, intencion

    # Fallback
    respuesta = (
        "No pude identificar tu solicitud. "
        "Puedo ayudarte con horarios, ubicación, servicios, "
        "citas, costos, métodos de pago, requisitos y contacto "
        "con recepción."
    )

    return respuesta, "fallback"

Celda 8. Ejecutar el chatbot

In [10]:
# ============================================================
# CHAT INTERACTIVO
# ============================================================

def iniciar_chat():

    print("=" * 65)
    print("                 GINEASSIST v1.0")
    print(" Chatbot de orientación para consultorio de ginecología")
    print("=" * 65)

    print(
        "\nGineAssist: Hola. Soy GineAssist. "
        "Puedo ayudarte con información general del consultorio."
    )

    print(
        "GineAssist: Puedes preguntarme sobre horarios, ubicación, "
        "servicios, citas, costos, pagos o requisitos."
    )

    print(
        "GineAssist: No realizo diagnósticos ni sustituyo "
        "la atención de un profesional de la salud."
    )

    print("\nEscribe 'salir' para terminar la conversación.\n")

    historial = []

    while True:

        usuario = input("Tú: ").strip()

        if normalizar_texto(usuario) in [
            "salir",
            "terminar",
            "adios",
            "bye"
        ]:
            print(
                "GineAssist: Gracias por utilizar GineAssist. "
                "Hasta luego."
            )
            break

        respuesta, intencion = generar_respuesta(usuario)

        print(f"GineAssist: {respuesta}\n")

        historial.append({
            "fecha": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "usuario": usuario,
            "intencion": intencion,
            "respuesta": respuesta
        })

    return historial

In [ ]:
historial = iniciar_chat()

                 GINEASSIST v1.0
 Chatbot de orientación para consultorio de ginecología

GineAssist: Hola. Soy GineAssist. Puedo ayudarte con información general del consultorio.
GineAssist: Puedes preguntarme sobre horarios, ubicación, servicios, citas, costos, pagos o requisitos.
GineAssist: No realizo diagnósticos ni sustituyo la atención de un profesional de la salud.

Escribe 'salir' para terminar la conversación.

Tú: Buenos dias
GineAssist: Hola. Soy GineAssist. ¿En qué puedo ayudarte?

Tú: a que horas cierran?
GineAssist: Nuestro horario de atención es lunes a jueves de 17:00 a 21:00.

Tú: donde se ubican?
GineAssist: No pude identificar tu solicitud. Puedo ayudarte con horarios, ubicación, servicios, citas, costos, métodos de pago, requisitos y contacto con recepción.

Tú: cual es su direccion?
GineAssist: El consultorio se encuentra en Centro Médico Puerta de Hierro Sur. Piso 7, consultorio 4A..

Tú: cuales son sus servicios?
GineAssist: Entre nuestros servicios se encuentra

Celda 9. Pruebas funcionales automaticas

In [12]:
# ============================================================
# PRUEBAS FUNCIONALES
# ============================================================

pruebas = [
    ("Hola buenos días", "saludo"),
    ("¿Cuál es su horario de atención?", "horario"),
    ("¿Dónde están ubicados?", "ubicacion"),
    ("¿Qué servicios ofrecen?", "servicios"),
    ("Quiero agendar una cita", "cita"),
    ("¿Cuánto cuesta la consulta?", "costo"),
    ("¿Aceptan tarjeta?", "pago"),
    ("¿Qué debo llevar a mi consulta?", "requisitos"),
    ("Quiero cambiar mi cita", "reprogramacion"),
    ("Quiero hablar con recepción", "contacto"),
    ("Necesito que interpretes mis resultados", "fuera_de_alcance"),
    ("Tengo una emergencia", "urgencia"),
    ("Quiero información sobre algo desconocido", "fallback")
]


resultados_pruebas = []

print("=" * 80)
print("PRUEBAS FUNCIONALES DE GINEASSIST")
print("=" * 80)

for numero, (pregunta, esperada) in enumerate(pruebas, start=1):

    respuesta, detectada = generar_respuesta(pregunta)

    correcto = esperada == detectada

    resultados_pruebas.append({
        "prueba": numero,
        "pregunta": pregunta,
        "esperada": esperada,
        "detectada": detectada,
        "correcto": correcto
    })

    print(f"\nPRUEBA {numero}")
    print(f"Usuario: {pregunta}")
    print(f"Intención esperada : {esperada}")
    print(f"Intención detectada: {detectada}")
    print(f"Resultado: {'CORRECTO' if correcto else 'INCORRECTO'}")
    print(f"GineAssist: {respuesta}")

PRUEBAS FUNCIONALES DE GINEASSIST

PRUEBA 1
Usuario: Hola buenos días
Intención esperada : saludo
Intención detectada: saludo
Resultado: CORRECTO
GineAssist: Hola. Soy GineAssist. ¿En qué puedo ayudarte?

PRUEBA 2
Usuario: ¿Cuál es su horario de atención?
Intención esperada : horario
Intención detectada: horario
Resultado: CORRECTO
GineAssist: Nuestro horario de atención es lunes a jueves de 17:00 a 21:00.

PRUEBA 3
Usuario: ¿Dónde están ubicados?
Intención esperada : ubicacion
Intención detectada: ubicacion
Resultado: CORRECTO
GineAssist: El consultorio se encuentra en Centro Médico Puerta de Hierro Sur. Piso 7, consultorio 4A..

PRUEBA 4
Usuario: ¿Qué servicios ofrecen?
Intención esperada : servicios
Intención detectada: servicios
Resultado: CORRECTO
GineAssist: Entre nuestros servicios se encuentran: consulta ginecológica, revisión ginecológica preventiva, orientación sobre servicios del consultorio. Para conocer si un servicio específico está disponible, puedes comunicarte con rece

Celda 10. Calcular resultados

In [13]:
# ============================================================
# EVALUACIÓN DE RESULTADOS
# ============================================================

total = len(resultados_pruebas)

correctas = sum(
    1 for resultado in resultados_pruebas
    if resultado["correcto"]
)

incorrectas = total - correctas

precision = correctas / total * 100

print("=" * 60)
print("RESULTADOS DE LAS PRUEBAS")
print("=" * 60)

print(f"Total de pruebas : {total}")
print(f"Correctas        : {correctas}")
print(f"Incorrectas      : {incorrectas}")
print(f"Exactitud        : {precision:.2f}%")

RESULTADOS DE LAS PRUEBAS
Total de pruebas : 13
Correctas        : 13
Incorrectas      : 0
Exactitud        : 100.00%


Celda 11. Tabla de resultados con Pandas

In [14]:
# ============================================================
# TABLA DE RESULTADOS
# ============================================================

import pandas as pd

df_resultados = pd.DataFrame(resultados_pruebas)

df_resultados["resultado"] = df_resultados["correcto"].apply(
    lambda x: "Correcto" if x else "Incorrecto"
)

df_resultados[
    [
        "prueba",
        "pregunta",
        "esperada",
        "detectada",
        "resultado"
    ]
]

,prueba,pregunta,esperada,detectada,resultado
0,1,Hola buenos días,saludo,saludo,Correcto
1,2,¿Cuál es su horario de atención?,horario,horario,Correcto
2,3,¿Dónde están ubicados?,ubicacion,ubicacion,Correcto
3,4,¿Qué servicios ofrecen?,servicios,servicios,Correcto
4,5,Quiero agendar una cita,cita,cita,Correcto
5,6,¿Cuánto cuesta la consulta?,costo,costo,Correcto
6,7,¿Aceptan tarjeta?,pago,pago,Correcto
7,8,¿Qué debo llevar a mi consulta?,requisitos,requisitos,Correcto
8,9,Quiero cambiar mi cita,reprogramacion,reprogramacion,Correcto
9,10,Quiero hablar con recepción,contacto,contacto,Correcto


Celda 12. Probar frases manualmente

In [15]:
# ============================================================
# FUNCIÓN PARA PRUEBAS MANUALES
# ============================================================

def probar(mensaje):

    respuesta, intencion = generar_respuesta(mensaje)

    print(f"Usuario: {mensaje}")
    print(f"Intención detectada: {intencion}")
    print(f"GineAssist: {respuesta}")


# Ejemplos
probar("¿A qué hora cierran?")
print()

probar("Necesito una cita")
print()

probar("¿Dónde se encuentran?")
print()

probar("¿Puedes interpretar mis resultados?")

Usuario: ¿A qué hora cierran?
Intención detectada: horario
GineAssist: Nuestro horario de atención es lunes a jueves de 17:00 a 21:00.

Usuario: Necesito una cita
Intención detectada: cita
GineAssist: Para solicitar una cita puedes comunicarte con recepción al 33-1655-1408.

Usuario: ¿Dónde se encuentran?
Intención detectada: ubicacion
GineAssist: El consultorio se encuentra en Centro Médico Puerta de Hierro Sur. Piso 7, consultorio 4A..

Usuario: ¿Puedes interpretar mis resultados?
Intención detectada: fuera_de_alcance
GineAssist: No puedo realizar diagnósticos, recomendar medicamentos, modificar tratamientos ni interpretar resultados médicos. Para este tipo de consulta debes comunicarte directamente con un profesional de la salud.
